# Quickstart: reliability metrics for NBCs and GeFs

This notebook shows how the models in `robustness_quantification` are used in the PGM'26 paper *Local Robustness Quantification for Naive Bayes Classifiers and Generative Forests: a General Approach*.

For a single dataset we
1. load and split the data,
2. fit a Naive Bayes Classifier (NBC) and a Generative Forest (GeF),
3. compute, for every test instance, uncertainty metrics (max. probability, entropy, aleatoric/epistemic/total uncertainty, margin of confidence) and robustness metrics (local $\epsilon$-contamination, total variation, $\chi^2$),
4. use each metric to reject the least reliable predictions and plot the resulting accuracy-rejection curves (ARCs).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from robustness_quantification.data import load_dataset
from robustness_quantification.data.splits import split_train_test
from robustness_quantification.evaluation.accrej_plots import get_accaccs_curve
from robustness_quantification.pipelines import standard_experiment_GeFs, standard_experiment_NBC

np.random.seed(0)

## 1. Data

`load_dataset` reads the (discretised) UCI datasets from the `data/` folder. `X` contains integer-coded categorical features, `C` is the number of classes and `F` the number of values of every feature.

In [ ]:
X, y, X_test, y_test, C, F = load_dataset('BCW', randomize=True)
X_train, y_train, X_test, y_test = split_train_test(X, y, X_test, y_test, fraction_test=0.4, max_size_test_set=500)
print(f'{len(y_train)} training and {len(y_test)} test instances, {C} classes, features with {F} values')

## 2. Fit the models and compute all reliability metrics

The `standard_experiment_*` functions fit the model, predict the test set and compute the requested metrics. They return a dictionary with one value per test instance for each metric, and a binary vector indicating which test predictions are correct.

In [ ]:
uncertainty_metrics = ['max_prob', 'entropies', 'aleatoric', 'epistemic', 'total', 'margin_of_confidence']
robustness_metrics = ['local_eps', 'chi2', 'total_var']
metrics = uncertainty_metrics + robustness_metrics

results = {}
results['NBC'] = standard_experiment_NBC(X_train, y_train, X_test, y_test, C, F, metrics, n_splits=5, N_bootstraps=10)
results['GeF'] = standard_experiment_GeFs(X_train, y_train, X_test, y_test, C, F, metrics,
                                          n_estimators=20, N_bootstraps=5, precision_bisearch=1e-9, max_iter=100)

for model, (rob_metrics, correct, _) in results.items():
    print(f'{model}: accuracy {np.mean(correct):.3f}')

## 3. Accuracy-rejection curves

Test instances are ranked by a metric (most reliable first). The ARC shows the accuracy on the instances that remain after rejecting the $n$ least reliable ones. A better metric yields a curve that rises faster; the area under the curve (AU-ARC) summarises this.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (model, (rob_metrics, correct, _)) in zip(axes, results.items()):
    for metric in metrics:
        curve = get_accaccs_curve(rob_metrics[metric], correct)
        ax.plot(curve, label=f'{metric} (AU-ARC {np.mean(curve):.4f})')
    ax.set_title(model)
    ax.set_xlabel('number of rejected instances')
axes[0].set_ylabel('accuracy on remaining instances')
axes[1].legend(fontsize=7, loc='lower right')
plt.tight_layout()
plt.show()